# Stage 10A: multi-scale GR/typewell alignment

This standalone Colab notebook audits a constrained structural correction against the verified ravaghi OOF base. It performs standard five-fold nested selection only and never submits to Kaggle. CPU high-RAM is recommended; a GPU is not used.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
data_dir = drive_root / 'data'
artifact_dir = drive_root / 'artifacts'
assert (data_dir / 'train').is_dir(), f'Missing competition data: {data_dir / "train"}'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)


## Reuse or build the Stage 8 candidate matrix

The large verified base is kept on Google Drive. If the Stage 8 matrix is absent but the Stage 7 base exists, this cell rebuilds it automatically.


In [ ]:
cutback_candidates = [
    artifact_dir / 'stage8_safe_cutback_gate_full_v002',
    artifact_dir / 'stage8_safe_cutback_gate_full_v001',
]
cutback_run = next((p for p in cutback_candidates if (p / 'candidate_matrix.parquet').is_file()), None)
if cutback_run is None:
    base_run = artifact_dir / 'stage7_public_residual_gate_full_v001'
    assert (base_run / 'base_oof.parquet').is_file() or (base_run / 'oof.parquet').is_file(), (
        'Stage 7 base artifact is missing. Run notebook 160 once; 00_colab_setup is not required.'
    )
    cutback_run = cutback_candidates[0]
    command = [
        'uv', 'run', 'rogii-safe-cutback',
        '--config', 'configs/experiment/stage8_safe_cutback_gate.yaml',
        '--base-run', str(base_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', cutback_run.name,
    ]
    subprocess.run(command, cwd=repo_dir, check=True)
print('Using:', cutback_run)


In [ ]:
RUN_ID = 'stage10_multiscale_alignment_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    command = [
        'uv', 'run', 'rogii-alignment',
        '--config', 'configs/experiment/stage10_multiscale_alignment.yaml',
        '--cutback-run', str(cutback_run), '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ]
    process = subprocess.Popen(command, cwd=repo_dir, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    if process.wait() != 0:
        raise RuntimeError(f'Stage 10 failed with exit code {process.returncode}')
else:
    print('Reusing completed run:', run_dir)


In [ ]:
summary = json.loads((run_dir / 'gate_summary.json').read_text())
{
    'promoted_to_spatial_audit': summary['promoted_to_spatial_audit'],
    'base_rmse': summary['base_metrics']['pooled_rmse'],
    'nested_candidate_rmse': summary['candidate_metrics']['pooled_rmse'],
    'rmse_delta': summary['pooled_rmse_delta'],
    'bootstrap_95pct': [summary['bootstrap']['ci_2_5'], summary['bootstrap']['ci_97_5']],
    'improved_folds': f"{summary['improved_folds']}/{len(summary['fold_deltas'])}",
    'fold_deltas': summary['fold_deltas'],
    'gates': summary['gates'],
    'inference_spec': summary['inference_spec'],
    'selections': summary['selections'],
    'spec_report': summary['spec_report'],
}
